# PodcastGen-Agent
Generate a complete podcast MP3 from any topic.

## Before you start
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run every cell in order.
3. The first generation downloads several large models and can take 10 to 25 minutes.
4. If Colab disconnects, reconnect and run the cells again. Completed voice segments are checkpointed.

In [ ]:
# Clone or update the repository first.
import os

REPO_URL = "https://github.com/Rashal10/PodcastGen-Agent.git"
REPO_DIR = "/content/PodcastGen-Agent"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    %cd {REPO_DIR}
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

print("Repository ready:", os.getcwd())

In [ ]:
# Install only Colab-compatible dependencies. This intentionally does not install
# Audiocraft because Audiocraft 1.3 requires Python 3.9 and torch 2.1.
import shutil
import subprocess
import sys

subprocess.check_call(["apt-get", "update", "-qq"])
subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg", "espeak-ng"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "TTS",
    "audiocraft",
])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    "requirements-colab.txt",
])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".",
    "--no-deps",
])

# Verify imports in a fresh Python subprocess so cached Colab modules cannot hide
# a broken installation.
verification = r'''
import shutil
import torch
import tokenizers
import transformers
from podcast_gen_agent.compat import ensure_transformers_compat

ensure_transformers_compat()

from TTS.api import TTS
from transformers import MusicgenForConditionalGeneration
from podcast_gen_agent.graph import get_graph

assert torch.cuda.is_available(), "GPU runtime is not enabled"
assert shutil.which("ffmpeg"), "ffmpeg is missing"
assert shutil.which("espeak-ng"), "espeak-ng is missing"
assert transformers.__version__ == "4.57.5", transformers.__version__
assert tokenizers.__version__ == "0.22.1", tokenizers.__version__
get_graph()
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("XTTS import: OK")
print("MusicGen import: OK")
print("LangGraph compile: OK")
print("ffmpeg and eSpeak: OK")
'''
subprocess.run([sys.executable, "-c", verification], check=True)
print("All dependencies and pipeline imports are ready.")

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Select Runtime → Change runtime type → T4 GPU, "
    "then run the notebook again."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Change only TOPIC and DURATION for each podcast.
import os
import subprocess
import sys
from pathlib import Path

from podcast_gen_agent.utils.output import find_latest_podcast

TOPIC = "LoRA and QLoRA"
DURATION = 5

assert TOPIC.strip(), "TOPIC must not be empty"
assert 1 <= DURATION <= 30, "DURATION must be between 1 and 30 minutes"

env = os.environ.copy()
env["COQUI_TOS_AGREED"] = "1"
env["PODCAST_LOG_LEVEL"] = "INFO"

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "podcast_gen_agent.main",
        TOPIC,
        "--duration",
        str(DURATION),
    ],
    env=env,
)
result.check_returncode()

latest = find_latest_podcast(Path("outputs"))
assert latest is not None and latest.stat().st_size > 0, (
    "Generation finished without producing a non-empty MP3. Check the logs above."
)
print(f"SUCCESS: {latest} ({latest.stat().st_size / 1_000_000:.1f} MB)")

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Playing: {latest}")
    display(Audio(str(latest)))
else:
    print("No podcast mp3 found. Check the generation cell for errors.")

In [ ]:
from pathlib import Path
from google.colab import files

from podcast_gen_agent.utils.output import find_latest_podcast

latest = find_latest_podcast(Path("outputs"))
if latest:
    print(f"Downloading: {latest}")
    files.download(str(latest))
else:
    print("No podcast file found under outputs/. Check the generation cell for errors.")